In [1]:
%matplotlib inline

from IPython.display import Markdown
from tqdm.notebook import trange

from src.algorithm.FixedRoutePolicy import FixedRoutePolicy
from src.algorithm.NearestUnservedPolicy import NearestUnservedPolicy
from src.algorithm.SkipStopFixedRoutePolicy import SkipStopFixedRoutePolicy
from src.algorithm.MarginalCostFilterPolicy import MarginalCostFilterPolicy
from src.experiment import (
    compare_policies,
    plot_kpi_bars,
    plot_occupancy,
    plot_wait_distribution,
    run_trial,
    show_animations,
    summarize_comparison,
)
from src.map.PlanarMapProvider import PlanarMapProvider
from src.map.graph import PlanarStreetGraph

In [2]:
MAP_SEED = 42

graph = PlanarStreetGraph.generate(
    n_seeds=17,
    bounds=(0.0, 0.0, 2000.0, 2000.0),
    seed=MAP_SEED,
    stop_density=0.5,
)

provider = PlanarMapProvider(graph)
print(f"{len(provider.get_stops())} bus stops on this map")

21 bus stops on this map


In [3]:
def param_sweep(graph, provider, start, end, step):
    N_TRIALS = 1000
    BASE_SEED = 0
    output = []
    
    for curr_max in trange(start, end, step):
        summary = summarize_comparison(
            policies=[
                ("Fixed route",      lambda g: FixedRoutePolicy(g)),
                ("Fixed route w/ skip", lambda g: SkipStopFixedRoutePolicy(g)),
                ("Nearest unserved", lambda g: NearestUnservedPolicy(g)),
                ("Marginal cost filter", lambda g: MarginalCostFilterPolicy(g))
                # ("Constrained insertion", lambda g: ConstrainedInsertionPolicy(g))
                
            ],
            graph=graph,
            n_trials=N_TRIALS,
            base_seed=BASE_SEED,
            # Forwarded to run_trial:
            max_requests=curr_max,
            capacity=40,
            max_wait_s=900.0
        )
        output.append(summary.markdown)

    return output

In [5]:
output1 = param_sweep(graph, provider, 50, 300, 50)

  0%|          | 0/5 [00:00<?, ?it/s]

Fixed route         :   0%|          | 0/1000 [00:00<?, ?it/s]

Fixed route w/ skip :   0%|          | 0/1000 [00:00<?, ?it/s]

Nearest unserved    :   0%|          | 0/1000 [00:00<?, ?it/s]

Marginal cost filter:   0%|          | 0/1000 [00:00<?, ?it/s]

Fixed route         :   0%|          | 0/1000 [00:00<?, ?it/s]

... (11 outputs left out)

Fixed route         :   0%|          | 0/1000 [00:00<?, ?it/s]

Fixed route w/ skip :   0%|          | 0/1000 [00:00<?, ?it/s]

Nearest unserved    :   0%|          | 0/1000 [00:00<?, ?it/s]

Marginal cost filter:   0%|          | 0/1000 [00:00<?, ?it/s]

In [16]:
Markdown(output1[0])

| Property | Fixed route | Fixed route w/ skip | Nearest unserved | Marginal cost filter |
| --- | --- | --- | --- | --- |
| Fulfilled requests | 34.7 (±0.6%) | 36.0 (±1.0%) | 37.6 (±0.9%) | 37.5 (±0.9%) |
| Cancelled requests | 15.3 (±1.3%) | 13.3 (±1.9%) | 11.7 (±1.8%) | 11.8 (±1.8%) |
| Mean wait time | 449.3 s (±0.6%) | 456.3 s (±0.9%) | 373.3 s (±1.0%) | 374.1 s (±1.0%) |
| P90 wait time | 790.1 s (±0.4%) | 778.0 s (±0.8%) | 668.5 s (±1.0%) | 670.6 s (±1.0%) |
| Mean ride time | 658.4 s (±0.6%) | 548.5 s (±1.0%) | 556.3 s (±1.6%) | 552.7 s (±1.6%) |
| Fulfillment rate | 69.3 % (±0.6%) | 73.3 % (±0.7%) | 76.6 % (±0.5%) | 76.4 % (±0.5%) |
| Served fraction | 69.3 % (±0.6%) | 73.3 % (±0.7%) | 76.6 % (±0.5%) | 76.4 % (±0.5%) |
| Throughput | 0.5 req/min (±0.7%) | 0.6 req/min (±4.3%) | 0.6 req/min (±4.0%) | 0.6 req/min (±4.0%) |
| Tick snapshots | 4442.3 (±0.4%) | 4084.4 (±0.8%) | 3997.8 (±0.8%) | 3993.3 (±0.8%) |
| Fulfilled (wait samples) | 34.7 (±0.6%) | 36.0 (±1.0%) | 37.6 (±0.9%) | 37.5 (±0.9%) |
| Min wait time | 20.7 s (±6.8%) | 27.4 s (±6.6%) | 25.0 s (±6.4%) | 24.6 s (±6.4%) |
| Max wait time | 875.2 s (±0.2%) | 862.4 s (±0.8%) | 832.4 s (±0.8%) | 831.4 s (±0.8%) |
| Min ride time | 38.2 s (±6.9%) | 34.4 s (±6.5%) | 16.6 s (±4.7%) | 16.8 s (±4.7%) |
| Max ride time | 1277.3 s (±0.2%) | 1185.5 s (±0.9%) | 2724.5 s (±1.6%) | 2692.2 s (±1.6%) |

In [17]:
Markdown(output1[1])

| Property | Fixed route | Fixed route w/ skip | Nearest unserved | Marginal cost filter |
| --- | --- | --- | --- | --- |
| Fulfilled requests | 68.7 (±0.4%) | 71.1 (±0.9%) | 74.2 (±0.8%) | 74.0 (±0.8%) |
| Cancelled requests | 31.3 (±0.9%) | 27.5 (±1.4%) | 24.4 (±1.3%) | 24.6 (±1.3%) |
| Mean wait time | 450.5 s (±0.4%) | 460.3 s (±0.8%) | 368.9 s (±0.9%) | 369.6 s (±0.9%) |
| P90 wait time | 801.1 s (±0.3%) | 790.7 s (±0.8%) | 675.5 s (±0.9%) | 676.1 s (±0.9%) |
| Mean ride time | 659.0 s (±0.4%) | 575.3 s (±0.9%) | 736.4 s (±1.5%) | 728.3 s (±1.5%) |
| Fulfillment rate | 68.7 % (±0.4%) | 72.5 % (±0.5%) | 75.6 % (±0.4%) | 75.4 % (±0.4%) |
| Served fraction | 68.7 % (±0.4%) | 72.5 % (±0.5%) | 75.6 % (±0.4%) | 75.4 % (±0.4%) |
| Throughput | 0.6 req/min (±0.4%) | 0.6 req/min (±3.8%) | 0.7 req/min (±3.5%) | 0.7 req/min (±3.5%) |
| Tick snapshots | 7430.4 (±0.2%) | 7057.2 (±0.8%) | 6955.5 (±0.8%) | 6950.2 (±0.8%) |
| Fulfilled (wait samples) | 68.7 (±0.4%) | 71.1 (±0.9%) | 74.2 (±0.8%) | 74.0 (±0.8%) |
| Min wait time | 11.4 s (±6.9%) | 13.1 s (±6.8%) | 11.9 s (±6.8%) | 12.0 s (±6.8%) |
| Max wait time | 888.8 s (±0.1%) | 874.8 s (±0.7%) | 855.0 s (±0.8%) | 854.6 s (±0.8%) |
| Min ride time | 17.6 s (±6.0%) | 16.9 s (±5.6%) | 11.0 s (±3.6%) | 11.0 s (±3.6%) |
| Max ride time | 1296.9 s (±0.1%) | 1249.4 s (±0.8%) | 5301.7 s (±1.4%) | 5202.6 s (±1.5%) |

In [18]:
Markdown(output1[2])

| Property | Fixed route | Fixed route w/ skip | Nearest unserved | Marginal cost filter |
| --- | --- | --- | --- | --- |
| Fulfilled requests | 102.9 (±0.4%) | 106.0 (±0.8%) | 110.6 (±0.8%) | 110.4 (±0.8%) |
| Cancelled requests | 47.1 (±0.8%) | 41.9 (±1.2%) | 37.3 (±1.2%) | 37.5 (±1.2%) |
| Mean wait time | 451.9 s (±0.3%) | 461.3 s (±0.8%) | 368.4 s (±0.8%) | 369.0 s (±0.8%) |
| P90 wait time | 804.9 s (±0.2%) | 794.1 s (±0.8%) | 678.6 s (±0.8%) | 679.5 s (±0.8%) |
| Mean ride time | 657.1 s (±0.3%) | 582.9 s (±0.8%) | 875.3 s (±1.5%) | 856.3 s (±1.5%) |
| Fulfillment rate | 68.6 % (±0.4%) | 72.1 % (±0.5%) | 75.1 % (±0.4%) | 75.0 % (±0.4%) |
| Served fraction | 68.6 % (±0.4%) | 72.1 % (±0.5%) | 75.1 % (±0.4%) | 75.0 % (±0.4%) |
| Throughput | 0.6 req/min (±0.4%) | 0.7 req/min (±3.5%) | 0.7 req/min (±3.3%) | 0.7 req/min (±3.4%) |
| Tick snapshots | 10421.0 (±0.2%) | 9987.4 (±0.7%) | 9910.7 (±0.7%) | 9914.1 (±0.7%) |
| Fulfilled (wait samples) | 102.9 (±0.4%) | 106.0 (±0.8%) | 110.6 (±0.8%) | 110.4 (±0.8%) |
| Min wait time | 7.9 s (±7.1%) | 8.8 s (±7.3%) | 7.9 s (±7.1%) | 8.2 s (±7.2%) |
| Max wait time | 892.4 s (±0.1%) | 878.5 s (±0.7%) | 863.4 s (±0.7%) | 864.7 s (±0.7%) |
| Min ride time | 12.2 s (±5.3%) | 12.2 s (±5.2%) | 9.0 s (±2.8%) | 8.9 s (±2.9%) |
| Max ride time | 1302.9 s (±0.1%) | 1272.6 s (±0.8%) | 7848.9 s (±1.4%) | 7582.8 s (±1.6%) |

In [19]:
Markdown(output1[3])

| Property | Fixed route | Fixed route w/ skip | Nearest unserved | Marginal cost filter |
| --- | --- | --- | --- | --- |
| Fulfilled requests | 137.1 (±0.3%) | 140.8 (±0.8%) | 147.0 (±0.8%) | 146.8 (±0.8%) |
| Cancelled requests | 62.9 (±0.7%) | 56.5 (±1.1%) | 50.2 (±1.1%) | 50.5 (±1.1%) |
| Mean wait time | 451.4 s (±0.3%) | 461.8 s (±0.8%) | 368.4 s (±0.8%) | 368.8 s (±0.8%) |
| P90 wait time | 807.1 s (±0.2%) | 796.2 s (±0.8%) | 681.5 s (±0.8%) | 682.1 s (±0.8%) |
| Mean ride time | 656.7 s (±0.3%) | 586.9 s (±0.8%) | 999.0 s (±1.5%) | 964.3 s (±1.5%) |
| Fulfillment rate | 68.6 % (±0.3%) | 71.8 % (±0.4%) | 74.9 % (±0.4%) | 74.8 % (±0.4%) |
| Served fraction | 68.6 % (±0.3%) | 71.8 % (±0.4%) | 74.9 % (±0.4%) | 74.8 % (±0.4%) |
| Throughput | 0.6 req/min (±0.3%) | 0.7 req/min (±3.4%) | 0.7 req/min (±3.3%) | 0.7 req/min (±3.3%) |
| Tick snapshots | 13420.6 (±0.1%) | 12951.7 (±0.7%) | 12871.9 (±0.7%) | 12869.7 (±0.7%) |
| Fulfilled (wait samples) | 137.1 (±0.3%) | 140.8 (±0.8%) | 147.0 (±0.8%) | 146.8 (±0.8%) |
| Min wait time | 6.7 s (±6.8%) | 6.4 s (±7.5%) | 5.8 s (±7.3%) | 5.9 s (±7.3%) |
| Max wait time | 893.5 s (±0.0%) | 880.5 s (±0.7%) | 869.3 s (±0.7%) | 870.0 s (±0.7%) |
| Min ride time | 9.7 s (±4.8%) | 9.9 s (±4.8%) | 8.1 s (±2.4%) | 8.0 s (±2.3%) |
| Max ride time | 1305.6 s (±0.0%) | 1286.5 s (±0.8%) | 10266.6 s (±1.5%) | 9788.4 s (±1.7%) |

In [20]:
Markdown(output1[4])

| Property | Fixed route | Fixed route w/ skip | Nearest unserved | Marginal cost filter |
| --- | --- | --- | --- | --- |
| Fulfilled requests | 171.2 (±0.3%) | 175.7 (±0.8%) | 183.4 (±0.8%) | 183.2 (±0.8%) |
| Cancelled requests | 78.8 (±0.6%) | 70.8 (±1.0%) | 63.2 (±1.0%) | 63.3 (±1.0%) |
| Mean wait time | 449.9 s (±0.3%) | 462.1 s (±0.8%) | 368.2 s (±0.8%) | 368.5 s (±0.8%) |
| P90 wait time | 806.0 s (±0.2%) | 797.8 s (±0.8%) | 681.8 s (±0.8%) | 682.2 s (±0.8%) |
| Mean ride time | 656.5 s (±0.3%) | 589.1 s (±0.8%) | 1111.6 s (±1.6%) | 1064.4 s (±1.6%) |
| Fulfillment rate | 68.5 % (±0.3%) | 71.7 % (±0.4%) | 74.7 % (±0.3%) | 74.7 % (±0.3%) |
| Served fraction | 68.5 % (±0.3%) | 71.7 % (±0.4%) | 74.7 % (±0.3%) | 74.7 % (±0.3%) |
| Throughput | 0.6 req/min (±0.3%) | 0.7 req/min (±3.4%) | 0.7 req/min (±3.2%) | 0.7 req/min (±3.2%) |
| Tick snapshots | 16414.9 (±0.1%) | 15913.9 (±0.7%) | 15834.0 (±0.7%) | 15828.8 (±0.7%) |
| Fulfilled (wait samples) | 171.2 (±0.3%) | 175.7 (±0.8%) | 183.4 (±0.8%) | 183.2 (±0.8%) |
| Min wait time | 4.3 s (±7.5%) | 5.0 s (±7.6%) | 4.4 s (±7.9%) | 4.5 s (±7.8%) |
| Max wait time | 894.3 s (±0.0%) | 881.7 s (±0.7%) | 872.5 s (±0.7%) | 873.0 s (±0.7%) |
| Min ride time | 8.3 s (±4.3%) | 8.5 s (±4.3%) | 7.6 s (±2.0%) | 7.5 s (±2.0%) |
| Max ride time | 1307.1 s (±0.0%) | 1294.4 s (±0.8%) | 12595.2 s (±1.6%) | 11907.2 s (±1.7%) |

In [22]:
output2 = param_sweep(graph, provider, 300, 600, 100)

  0%|          | 0/3 [00:00<?, ?it/s]

Fixed route         :   0%|          | 0/1000 [00:00<?, ?it/s]

Fixed route w/ skip :   0%|          | 0/1000 [00:00<?, ?it/s]

Nearest unserved    :   0%|          | 0/1000 [00:00<?, ?it/s]

Marginal cost filter:   0%|          | 0/1000 [00:00<?, ?it/s]

Fixed route         :   0%|          | 0/1000 [00:00<?, ?it/s]

... (3 outputs left out)

Fixed route         :   0%|          | 0/1000 [00:00<?, ?it/s]

Fixed route w/ skip :   0%|          | 0/1000 [00:00<?, ?it/s]

Nearest unserved    :   0%|          | 0/1000 [00:00<?, ?it/s]

Marginal cost filter:   0%|          | 0/1000 [00:00<?, ?it/s]

In [23]:
Markdown(output2[0])

| Property | Fixed route | Fixed route w/ skip | Nearest unserved | Marginal cost filter |
| --- | --- | --- | --- | --- |
| Fulfilled requests | 205.6 (±0.2%) | 210.8 (±0.8%) | 219.9 (±0.8%) | 219.4 (±0.8%) |
| Cancelled requests | 94.4 (±0.5%) | 85.0 (±1.0%) | 75.9 (±1.0%) | 76.4 (±1.0%) |
| Mean wait time | 449.5 s (±0.2%) | 462.2 s (±0.8%) | 367.9 s (±0.8%) | 368.3 s (±0.8%) |
| P90 wait time | 806.6 s (±0.1%) | 798.6 s (±0.7%) | 682.4 s (±0.8%) | 683.0 s (±0.8%) |
| Mean ride time | 657.0 s (±0.2%) | 591.4 s (±0.8%) | 1216.2 s (±1.6%) | 1157.9 s (±1.7%) |
| Fulfillment rate | 68.5 % (±0.2%) | 71.7 % (±0.4%) | 74.7 % (±0.3%) | 74.5 % (±0.3%) |
| Served fraction | 68.5 % (±0.2%) | 71.7 % (±0.4%) | 74.7 % (±0.3%) | 74.5 % (±0.3%) |
| Throughput | 0.6 req/min (±0.2%) | 0.7 req/min (±3.3%) | 0.7 req/min (±3.2%) | 0.7 req/min (±3.2%) |
| Tick snapshots | 19450.9 (±0.1%) | 18875.3 (±0.7%) | 18793.4 (±0.7%) | 18794.0 (±0.7%) |
| Fulfilled (wait samples) | 205.6 (±0.2%) | 210.8 (±0.8%) | 219.9 (±0.8%) | 219.4 (±0.8%) |
| Min wait time | 3.2 s (±8.0%) | 3.9 s (±8.1%) | 3.4 s (±8.1%) | 3.7 s (±8.1%) |
| Max wait time | 895.1 s (±0.0%) | 882.3 s (±0.7%) | 874.9 s (±0.7%) | 875.4 s (±0.7%) |
| Min ride time | 7.4 s (±3.8%) | 7.6 s (±4.0%) | 7.3 s (±1.9%) | 7.2 s (±1.9%) |
| Max ride time | 1307.9 s (±0.0%) | 1301.0 s (±0.8%) | 14815.3 s (±1.6%) | 13936.9 s (±1.8%) |

In [24]:
Markdown(output2[1])

| Property | Fixed route | Fixed route w/ skip | Nearest unserved | Marginal cost filter |
| --- | --- | --- | --- | --- |
| Fulfilled requests | 274.2 (±0.2%) | 280.5 (±0.8%) | 292.1 (±0.8%) | 292.0 (±0.8%) |
| Cancelled requests | 125.8 (±0.4%) | 113.9 (±0.9%) | 102.3 (±0.9%) | 102.5 (±0.9%) |
| Mean wait time | 449.9 s (±0.2%) | 462.5 s (±0.8%) | 368.2 s (±0.8%) | 368.6 s (±0.8%) |
| P90 wait time | 807.5 s (±0.1%) | 799.4 s (±0.7%) | 684.2 s (±0.8%) | 684.9 s (±0.8%) |
| Mean ride time | 657.2 s (±0.2%) | 593.5 s (±0.8%) | 1393.8 s (±1.8%) | 1303.3 s (±1.8%) |
| Fulfillment rate | 68.5 % (±0.2%) | 71.5 % (±0.4%) | 74.4 % (±0.3%) | 74.4 % (±0.3%) |
| Served fraction | 68.5 % (±0.2%) | 71.5 % (±0.4%) | 74.4 % (±0.3%) | 74.4 % (±0.3%) |
| Throughput | 0.6 req/min (±0.2%) | 0.7 req/min (±3.3%) | 0.7 req/min (±3.1%) | 0.7 req/min (±3.1%) |
| Tick snapshots | 25437.6 (±0.1%) | 24787.4 (±0.7%) | 24720.9 (±0.7%) | 24711.8 (±0.7%) |
| Fulfilled (wait samples) | 274.2 (±0.2%) | 280.5 (±0.8%) | 292.1 (±0.8%) | 292.0 (±0.8%) |
| Min wait time | 2.0 s (±8.6%) | 2.8 s (±8.4%) | 2.4 s (±8.9%) | 2.6 s (±8.9%) |
| Max wait time | 896.4 s (±0.0%) | 883.5 s (±0.7%) | 877.6 s (±0.7%) | 877.9 s (±0.7%) |
| Min ride time | 6.3 s (±3.2%) | 6.5 s (±3.4%) | 6.9 s (±1.7%) | 6.9 s (±1.7%) |
| Max ride time | 1308.8 s (±0.0%) | 1311.3 s (±0.8%) | 18879.1 s (±1.7%) | 17551.5 s (±1.9%) |

In [25]:
Markdown(output2[2])

| Property | Fixed route | Fixed route w/ skip | Nearest unserved | Marginal cost filter |
| --- | --- | --- | --- | --- |
| Fulfilled requests | 342.6 (±0.2%) | 350.5 (±0.8%) | 363.2 (±0.8%) | 363.2 (±0.8%) |
| Cancelled requests | 157.4 (±0.4%) | 142.6 (±0.9%) | 129.8 (±0.9%) | 129.8 (±0.9%) |
| Mean wait time | 449.9 s (±0.2%) | 462.5 s (±0.8%) | 369.4 s (±0.8%) | 369.7 s (±0.8%) |
| P90 wait time | 807.5 s (±0.1%) | 800.0 s (±0.7%) | 686.7 s (±0.8%) | 687.7 s (±0.8%) |
| Mean ride time | 657.3 s (±0.2%) | 594.9 s (±0.8%) | 1486.1 s (±1.7%) | 1394.8 s (±1.8%) |
| Fulfillment rate | 68.5 % (±0.2%) | 71.5 % (±0.4%) | 74.0 % (±0.3%) | 74.0 % (±0.3%) |
| Served fraction | 68.5 % (±0.2%) | 71.5 % (±0.4%) | 74.0 % (±0.3%) | 74.0 % (±0.3%) |
| Throughput | 0.7 req/min (±0.2%) | 0.7 req/min (±3.3%) | 0.8 req/min (±3.1%) | 0.8 req/min (±3.1%) |
| Tick snapshots | 31438.5 (±0.1%) | 30715.9 (±0.7%) | 30634.8 (±0.7%) | 30634.8 (±0.7%) |
| Fulfilled (wait samples) | 342.6 (±0.2%) | 350.5 (±0.8%) | 363.2 (±0.8%) | 363.2 (±0.8%) |
| Min wait time | 1.7 s (±8.9%) | 2.1 s (±9.4%) | 1.9 s (±9.8%) | 2.0 s (±10.1%) |
| Max wait time | 896.7 s (±0.0%) | 884.2 s (±0.7%) | 879.6 s (±0.7%) | 879.7 s (±0.7%) |
| Min ride time | 5.6 s (±2.7%) | 5.8 s (±3.1%) | 6.7 s (±1.5%) | 6.6 s (±1.6%) |
| Max ride time | 1309.3 s (±0.0%) | 1319.0 s (±0.8%) | 21884.3 s (±1.8%) | 20333.6 s (±2.0%) |